# Global K-Means On Saved PCA-Wavelet-PCA Projections

This notebook starts from already generated projection files and fits one shared k-means model across all species. It is self-contained apart from the saved projection `.npy` files and standard scientific Python packages.

In [5]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans

REPO_ROOT = Path('/Users/meganbishop/slowmodeevo').resolve()

def find_projection_run_root():
    base = REPO_ROOT / 'outputs/single_species_distance_comparison'
    candidates = [base / value for value in ['pca_multispecies_evensamp' ,'multispecies_pca_wavelet_pca/pca_before_wavelet_pca_after_multispecies', 'pca_before_wavelet_pca_after_multispecies']]
    candidates.extend(
        sorted(
            [path for path in base.glob('**/projection_files_manifest.csv')],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )[index].parent
        for index in range(len(list(base.glob('**/projection_files_manifest.csv'))))
    )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        required = [
            candidate / 'projection_generation_summary.csv',
            candidate / 'multispecies_individual_manifest.csv',
            candidate / 'projection_files_manifest.csv',
        ]
        if all(path.exists() for path in required):
            return candidate
    raise FileNotFoundError(
        'Could not find a completed multispecies projection run under '
        f'{base}. Expected projection_generation_summary.csv, '
        'multispecies_individual_manifest.csv, and projection_files_manifest.csv.'
    )

PROJECTION_RUN_ROOT = find_projection_run_root()

N_CLUSTERS = 1000
D_EMBED = 10
KMEANS_SUBSAMPLE_FACTOR = 20
KMEANS_N_INIT = 20
SEED = 14
REUSE_EXISTING = True
CHUNK_SIZE = 200_000
PARTIAL_FIT_BATCH_ROWS = 100_000

# Fit the shared codebook from equal numbers of delay-embedded samples per species.
# This prevents high-frame-count species from defining more global centroids just
# because they contribute more recordings/frames to the projection manifest.
BALANCE_KMEANS_SAMPLES_BY_SPECIES = True
BALANCED_ROWS_PER_SPECIES = None  # None uses the smallest species' fixed-stride sample count.
KMEANS_FIT_LABEL = 'balanced_species' if BALANCE_KMEANS_SAMPLES_BY_SPECIES else 'raw_streaming'
KMEANS_ROOT = PROJECTION_RUN_ROOT / f'global_kmeans_N{N_CLUSTERS}_{KMEANS_FIT_LABEL}'
KMEANS_ROOT.mkdir(parents=True, exist_ok=True)

PROJECTION_MANIFEST_CSV = PROJECTION_RUN_ROOT / 'projection_files_manifest.csv'
projection_manifest = pd.read_csv(PROJECTION_MANIFEST_CSV)
proj_files = projection_manifest['projection_file'].astype(str).tolist()
print(f'PROJECTION_RUN_ROOT = {PROJECTION_RUN_ROOT}')
print(f'KMEANS_ROOT = {KMEANS_ROOT}')
print(f'Loaded {len(proj_files)} projection files from {PROJECTION_MANIFEST_CSV}')
display(projection_manifest.groupby('species')['n_frames'].agg(['count', 'sum']).reset_index())


PROJECTION_RUN_ROOT = /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp
KMEANS_ROOT = /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp/global_kmeans_N1000_balanced_species
Loaded 354 projection files from /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp/projection_files_manifest.csv


,species,count,sum
0,Mus_caroli,15,9288344
1,Mus_musculus,93,85754643
2,Mus_spretus,16,9720360
3,Peromyscus_californicus,23,5183880
4,Peromyscus_gossypinus,20,4319900
5,Peromyscus_leucopus,14,3023930
6,Peromyscus_maniculatus,129,31751265
7,Peromyscus_polionotus,44,10583755


## Fit Shared N=1000 K-Means And Save States

In [7]:
def delay_embed_array(x, d):
    x = np.asarray(x)
    d = int(d)
    if d <= 1:
        return np.asarray(x, dtype=np.float32)
    if x.shape[0] < d:
        return np.empty((0, x.shape[1] * d), dtype=np.float32)
    windows = [x[offset: x.shape[0] - d + offset + 1] for offset in range(d)]
    return np.concatenate(windows, axis=1).astype(np.float32, copy=False)


def delay_embed_at_starts(x, d, starts):
    x = np.asarray(x)
    starts = np.asarray(starts, dtype=int)
    if starts.size == 0:
        return np.empty((0, x.shape[1] * int(d)), dtype=np.float32)
    pieces = [x[starts + offset] for offset in range(int(d))]
    return np.concatenate(pieces, axis=1).astype(np.float32, copy=False)


def delay_embed_subsample(x, d, factor):
    x = np.asarray(x)
    d = int(d)
    factor = max(1, int(factor))
    starts = np.arange(0, max(x.shape[0] - d + 1, 0), factor, dtype=int)
    return delay_embed_at_starts(x, d, starts)


def embedded_row_count(n_frames, d):
    return max(int(n_frames) - int(d) + 1, 0)


def fixed_stride_sample_count(n_frames, d, factor):
    n_embedded = embedded_row_count(n_frames, d)
    if n_embedded == 0:
        return 0
    return int((n_embedded - 1) // max(1, int(factor)) + 1)


def build_balanced_sample_plan(projection_manifest, d, factor, rows_per_species=None):
    plan = projection_manifest.reset_index(drop=True).copy()
    plan['row_index'] = np.arange(len(plan))
    plan['n_embedded_rows'] = plan['n_frames'].map(lambda n: embedded_row_count(n, d)).astype(int)
    plan['fixed_stride_sample_rows'] = plan['n_frames'].map(lambda n: fixed_stride_sample_count(n, d, factor)).astype(int)
    species_stride_rows = plan.groupby('species')['fixed_stride_sample_rows'].sum().sort_index()
    if rows_per_species is None:
        target_rows = int(species_stride_rows.min())
    else:
        target_rows = int(rows_per_species)
    target_rows = max(1, target_rows)
    plan['species_sample_budget'] = plan['species'].map(lambda s: min(target_rows, int(species_stride_rows.loc[s]))).astype(int)
    plan['balanced_sample_rows'] = 0
    for species, sub in plan.groupby('species', sort=True):
        species_total = int(sub['fixed_stride_sample_rows'].sum())
        species_budget = min(target_rows, species_total)
        if species_total <= 0 or species_budget <= 0:
            continue
        exact = sub['fixed_stride_sample_rows'].to_numpy(float) * species_budget / species_total
        take = np.floor(exact).astype(int)
        remainders = exact - take
        shortfall = int(species_budget - take.sum())
        if shortfall > 0:
            order = np.argsort(-remainders, kind='mergesort')[:shortfall]
            take[order] += 1
        take = np.minimum(take, sub['fixed_stride_sample_rows'].to_numpy(int))
        # If clipping somehow left a shortfall, add rows to individuals with spare sampled frames.
        shortfall = int(species_budget - take.sum())
        if shortfall > 0:
            spare = sub['fixed_stride_sample_rows'].to_numpy(int) - take
            for local_idx in np.argsort(-spare, kind='mergesort'):
                if shortfall <= 0 or spare[local_idx] <= 0:
                    break
                add = min(shortfall, int(spare[local_idx]))
                take[local_idx] += add
                shortfall -= add
        plan.loc[sub.index, 'balanced_sample_rows'] = take
    summary = pd.DataFrame({
        'species': species_stride_rows.index,
        'fixed_stride_sample_rows': species_stride_rows.to_numpy(dtype=int),
        'balanced_sample_budget': [min(target_rows, int(value)) for value in species_stride_rows],
    })
    summary['actual_balanced_sample_rows'] = summary['species'].map(
        plan.groupby('species')['balanced_sample_rows'].sum()
    ).astype(int).to_numpy()
    summary['retained_fraction_of_fixed_stride_sample'] = np.divide(
        summary['actual_balanced_sample_rows'],
        np.maximum(summary['fixed_stride_sample_rows'], 1),
    )
    return plan, summary, target_rows


sample_plan, balanced_sample_summary, balanced_rows_per_species = build_balanced_sample_plan(
    projection_manifest,
    D_EMBED,
    KMEANS_SUBSAMPLE_FACTOR,
    BALANCED_ROWS_PER_SPECIES,
)
sample_plan.to_csv(KMEANS_ROOT / 'global_kmeans_balanced_sample_plan.csv', index=False)
balanced_sample_summary.to_csv(KMEANS_ROOT / 'global_kmeans_balanced_sample_summary.csv', index=False)
print(f'Balanced K-means rows per species: {balanced_rows_per_species:,}')
display(balanced_sample_summary)


def iter_projection_samples(projection_manifest, d, factor):
    for row_index, row in projection_manifest.reset_index(drop=True).iterrows():
        proj_file = row['projection_file']
        proj = np.load(proj_file, mmap_mode='r')
        sample = delay_embed_subsample(proj, d, factor)
        print(f'[{row_index + 1}/{len(projection_manifest)}] sampled {Path(proj_file).name}: {sample.shape}')
        yield row, sample
        del proj, sample


def iter_balanced_projection_samples(sample_plan, d, factor, seed):
    rng = np.random.default_rng(int(seed))
    plan = sample_plan.loc[sample_plan['balanced_sample_rows'] > 0].copy()
    plan['_shuffle_key'] = rng.random(len(plan))
    plan = plan.sort_values('_shuffle_key').drop(columns='_shuffle_key')
    taken_by_species = {str(species): 0 for species in sorted(plan['species'].astype(str).unique())}
    for _, row in plan.iterrows():
        proj_file = row['projection_file']
        n_embedded = int(row['n_embedded_rows'])
        n_take = int(row['balanced_sample_rows'])
        stride_starts = np.arange(0, n_embedded, max(1, int(factor)), dtype=int)
        if stride_starts.size == 0 or n_take <= 0:
            continue
        if n_take < stride_starts.size:
            starts = np.sort(rng.choice(stride_starts, size=n_take, replace=False))
        else:
            starts = stride_starts
        proj = np.load(proj_file, mmap_mode='r')
        sample = delay_embed_at_starts(proj, d, starts)
        species = str(row['species'])
        taken_by_species[species] = taken_by_species.get(species, 0) + int(sample.shape[0])
        print(
            f'[{species}] sampled {Path(proj_file).name}: {sample.shape}; '
            f'species rows so far={taken_by_species[species]:,}'
        )
        yield row, sample
        del proj, sample
    expected_by_species = sample_plan.groupby('species')['balanced_sample_rows'].sum().astype(int)
    mismatches = {
        str(species): (int(expected), int(taken_by_species.get(str(species), 0)))
        for species, expected in expected_by_species.items()
        if int(expected) != int(taken_by_species.get(str(species), 0))
    }
    if mismatches:
        raise RuntimeError(f'Balanced sample rows did not match plan: {mismatches}')


def flush_fit_buffer(kmeans, fit_buffer, *, force=False):
    if not fit_buffer:
        return 0
    buffered_rows = sum(block.shape[0] for block in fit_buffer)
    min_rows = int(kmeans.n_clusters)
    if buffered_rows < min_rows and not force:
        return 0
    if buffered_rows < min_rows:
        raise ValueError(
            f'Final MiniBatchKMeans buffer has {buffered_rows} rows, '
            f'but n_clusters={min_rows}. Increase balanced sampling rows or lower N_CLUSTERS.'
        )
    batch = np.concatenate(fit_buffer, axis=0).astype(np.float32, copy=False)
    fit_buffer.clear()
    n_fit = int(batch.shape[0])
    kmeans.partial_fit(batch)
    del batch
    return n_fit


def partial_fit_in_row_batches(kmeans, sample, batch_rows, fit_buffer=None):
    n_rows = int(sample.shape[0])
    fit_rows = 0
    if fit_buffer is None:
        fit_buffer = []
    for start in range(0, n_rows, int(batch_rows)):
        stop = min(start + int(batch_rows), n_rows)
        batch = np.asarray(sample[start:stop], dtype=np.float32)
        if batch.shape[0]:
            fit_buffer.append(batch)
            fit_rows += flush_fit_buffer(kmeans, fit_buffer, force=False)
    return fit_rows


def score_inertia_in_row_batches(kmeans, sample, batch_rows):
    inertia = 0.0
    n_rows = int(sample.shape[0])
    for start in range(0, n_rows, int(batch_rows)):
        stop = min(start + int(batch_rows), n_rows)
        batch = np.asarray(sample[start:stop], dtype=np.float32)
        if batch.shape[0]:
            distances = kmeans.transform(batch)
            inertia += float(np.square(np.min(distances, axis=1)).sum())
    return inertia


def iter_kmeans_fit_samples(seed):
    if BALANCE_KMEANS_SAMPLES_BY_SPECIES:
        return iter_balanced_projection_samples(sample_plan, D_EMBED, KMEANS_SUBSAMPLE_FACTOR, seed)
    return iter_projection_samples(projection_manifest, D_EMBED, KMEANS_SUBSAMPLE_FACTOR)


def fit_streaming_minibatch_kmeans():
    best_kmeans = None
    best_inertia = np.inf
    fit_rows = 0
    run_rows = []
    for init_index in range(KMEANS_N_INIT):
        print()
        print(f'MiniBatchKMeans init {init_index + 1}/{KMEANS_N_INIT}')
        kmeans = MiniBatchKMeans(
            n_clusters=N_CLUSTERS,
            batch_size=max(N_CLUSTERS * 5, 1000),
            n_init=1,
            random_state=SEED + init_index,
            init='random',
        )
        total_rows = 0
        sample_seed = SEED + 10_000 * init_index
        fit_buffer = []
        for _, sample in iter_kmeans_fit_samples(sample_seed):
            total_rows += int(sample.shape[0])
            partial_fit_in_row_batches(kmeans, sample, PARTIAL_FIT_BATCH_ROWS, fit_buffer)
        flush_fit_buffer(kmeans, fit_buffer, force=True)
        inertia = 0.0
        for _, sample in iter_kmeans_fit_samples(sample_seed):
            inertia += score_inertia_in_row_batches(kmeans, sample, PARTIAL_FIT_BATCH_ROWS)
        run_rows.append({
            'init_index': init_index,
            'random_state': SEED + init_index,
            'sample_seed': sample_seed,
            'sample_rows': total_rows,
            'balanced_by_species': bool(BALANCE_KMEANS_SAMPLES_BY_SPECIES),
            'balanced_rows_per_species': int(balanced_rows_per_species) if BALANCE_KMEANS_SAMPLES_BY_SPECIES else np.nan,
            'streaming_inertia': inertia,
        })
        print(f'init {init_index + 1}: rows={total_rows:,}, inertia={inertia:.3g}')
        fit_rows = max(fit_rows, total_rows)
        if inertia < best_inertia:
            best_inertia = inertia
            best_kmeans = kmeans
    pd.DataFrame(run_rows).to_csv(KMEANS_ROOT / 'global_kmeans_streaming_init_scores.csv', index=False)
    return best_kmeans, int(fit_rows), float(best_inertia)


def predict_delay_embedded_in_chunks(kmeans, proj, d, chunk_size=200_000):
    n_embedded = max(int(proj.shape[0]) - int(d) + 1, 0)
    states = np.empty(n_embedded, dtype=np.int32)
    for start in range(0, n_embedded, int(chunk_size)):
        stop = min(start + int(chunk_size), n_embedded)
        block = np.asarray(proj[start: stop + int(d) - 1], dtype=np.float32)
        embedded = delay_embed_array(block, d)
        states[start:stop] = kmeans.predict(embedded).astype(np.int32, copy=False)
    return states

manifest_path = KMEANS_ROOT / 'global_kmeans_manifest.csv'
result_path = KMEANS_ROOT / 'global_kmeans_result.pkl'
if REUSE_EXISTING and manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print(f'Reusing saved manifest: {manifest_path}')
else:
    kmeans, fit_rows, best_inertia = fit_streaming_minibatch_kmeans()
    print(f'best streaming k-means inertia={best_inertia:.3g}; sampled rows per init={fit_rows:,}')

    state_dir = KMEANS_ROOT / 'states'
    state_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, row in projection_manifest.iterrows():
        proj_file = row['projection_file']
        global_id = row['global_id']
        print(f'[{i + 1}/{len(projection_manifest)}] assigning {global_id}')
        proj = np.load(proj_file, mmap_mode='r')
        states = predict_delay_embedded_in_chunks(kmeans, proj, D_EMBED, CHUNK_SIZE)
        state_file = state_dir / f'{global_id}_states.npy'
        np.save(state_file, states)
        rows.append({
            'global_id': global_id,
            'species': row['species'],
            'source_individual_id': row['source_individual_id'],
            'projection_file': proj_file,
            'state_file': str(state_file.resolve()),
            'n_embedded_frames': int(states.shape[0]),
            'd_embed': D_EMBED,
            'N': N_CLUSTERS,
            'kmeans_subsample_factor': KMEANS_SUBSAMPLE_FACTOR,
            'kmeans_fit_mode': 'balanced_species_streaming_partial_fit' if BALANCE_KMEANS_SAMPLES_BY_SPECIES else 'streaming_partial_fit',
            'kmeans_balanced_by_species': bool(BALANCE_KMEANS_SAMPLES_BY_SPECIES),
            'kmeans_balanced_rows_per_species': int(balanced_rows_per_species) if BALANCE_KMEANS_SAMPLES_BY_SPECIES else np.nan,
            'kmeans_fit_rows_per_init': fit_rows,
            'kmeans_best_streaming_inertia': best_inertia,
        })
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False)
    with open(result_path, 'wb') as handle:
        pickle.dump({
            'kmeans': kmeans,
            'd_embed': D_EMBED,
            'N': N_CLUSTERS,
            'kmeans_subsample_factor': KMEANS_SUBSAMPLE_FACTOR,
            'kmeans_fit_mode': 'balanced_species_streaming_partial_fit' if BALANCE_KMEANS_SAMPLES_BY_SPECIES else 'streaming_partial_fit',
            'kmeans_balanced_by_species': bool(BALANCE_KMEANS_SAMPLES_BY_SPECIES),
            'kmeans_balanced_rows_per_species': int(balanced_rows_per_species) if BALANCE_KMEANS_SAMPLES_BY_SPECIES else None,
            'kmeans_fit_rows_per_init': fit_rows,
            'kmeans_best_streaming_inertia': best_inertia,
            'projection_manifest': str(PROJECTION_MANIFEST_CSV),
            'manifest': str(manifest_path),
        }, handle)
    print(f'Saved global k-means output under {KMEANS_ROOT}')

display(manifest.groupby('species')['n_embedded_frames'].agg(['count', 'sum']).reset_index())


Balanced K-means rows per species: 151,200


,species,fixed_stride_sample_rows,balanced_sample_budget,actual_balanced_sample_rows,retained_fraction_of_fixed_stride_sample
0,Mus_caroli,464415,151200,151200,0.325571
1,Mus_musculus,4287723,151200,151200,0.035263
2,Mus_spretus,486016,151200,151200,0.311101
3,Peromyscus_californicus,259200,151200,151200,0.583333
4,Peromyscus_gossypinus,216000,151200,151200,0.700000
5,Peromyscus_leucopus,151200,151200,151200,1.000000
6,Peromyscus_maniculatus,1587597,151200,151200,0.095238
7,Peromyscus_polionotus,529200,151200,151200,0.285714


Reusing saved manifest: /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp/global_kmeans_N1000_balanced_species/global_kmeans_manifest.csv


,species,count,sum
0,Mus_caroli,15,9288209
1,Mus_musculus,93,85753806
2,Mus_spretus,16,9720216
3,Peromyscus_californicus,23,5183673
4,Peromyscus_gossypinus,20,4319720
5,Peromyscus_leucopus,14,3023804
6,Peromyscus_maniculatus,129,31750104
7,Peromyscus_polionotus,44,10583359


## Quick Occupancy Check

In [8]:
counts = np.zeros(N_CLUSTERS, dtype=np.int64)
for state_file in manifest['state_file']:
    states = np.load(state_file, mmap_mode='r')
    counts += np.bincount(states.astype(int), minlength=N_CLUSTERS)
occupancy = pd.DataFrame({
    'cluster': np.arange(N_CLUSTERS),
    'n_frames': counts,
    'fraction': counts / max(counts.sum(), 1),
}).sort_values('n_frames', ascending=False)
occupancy.to_csv(KMEANS_ROOT / 'global_kmeans_cluster_occupancy.csv', index=False)
display(occupancy.head(20))

,cluster,n_frames,fraction
198,198,711424,0.004457
869,869,603086,0.003778
985,985,555728,0.003482
116,116,546136,0.003421
981,981,541410,0.003392
112,112,531164,0.003328
215,215,528726,0.003312
585,585,503036,0.003151
891,891,491180,0.003077
528,528,484000,0.003032


In [10]:
TRANSFER_MANIFEST = manifest.copy()

In [11]:
TRANSFER_MANIFEST 

,global_id,species,source_individual_id,projection_file,state_file,n_embedded_frames,d_embed,N,kmeans_subsample_factor,kmeans_fit_mode,kmeans_balanced_by_species,kmeans_balanced_rows_per_species,kmeans_fit_rows_per_init,kmeans_best_streaming_inertia
0,Mus_caroli__subject__CAROLI-F-2L-741,Mus_caroli,CAROLI-F-2L-741,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,432007,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
1,Mus_caroli__subject__CAROLI-F-L-737,Mus_caroli,CAROLI-F-L-737,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,648015,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
2,Mus_caroli__subject__CAROLI-F-L-741,Mus_caroli,CAROLI-F-L-741,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,648015,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
3,Mus_caroli__subject__CAROLI-F-LR-741,Mus_caroli,CAROLI-F-LR-741,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,648015,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
4,Mus_caroli__subject__CAROLI-F-N-740,Mus_caroli,CAROLI-F-N-740,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,648015,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,Peromyscus_polionotus__subject__40386.0,Peromyscus_polionotus,40386.0,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,215986,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
350,Peromyscus_polionotus__subject__40414.0,Peromyscus_polionotus,40414.0,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,215986,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
351,Peromyscus_polionotus__subject__40415.0,Peromyscus_polionotus,40415.0,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,215986,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
352,Peromyscus_polionotus__subject__40416.0,Peromyscus_polionotus,40416.0,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...,215986,10,1000,20,balanced_species_streaming_partial_fit,True,151200,1209600,8880.027203
